# A2: From Trees to Neural Networks
## Home Credit Default Risk Pipeline

---

## Phase 0: Setup

In [ ]:
! pip install ipykernel matplotlib seaborn scikit-learn xgboost kaggle

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
import zipfile

from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV,
    cross_val_score, learning_curve, validation_curve
)
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, precision_recall_curve,
    confusion_matrix, classification_report, roc_curve
)

from xgboost import XGBClassifier
import xgboost as xgb

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

---

## Phase 1: Data Preparation

**Dataset:** [Home Credit Default Risk](https://www.kaggle.com/c/home-credit-default-risk)  
**Task:** Predict whether an applicant will repay a loan (TARGET: 0 = repay, 1 = default).

**Key requirements:**
- Handle missing values and inconsistent types
- Apply label encoding or one-hot encoding to categorical variables
- Optionally construct new features (e.g., ratios, income-to-credit, age groups)
- Split into train / validation / test (70 / 15 / 15)
- For MLP: apply feature scaling (standardization). Discuss why necessary for NNs but not trees.
- **No data leakage:** All preprocessing fit on training set only.

In [ ]:
# --- Step 1.1: Load the Home Credit Default Risk dataset ---
# Option A: Download via Kaggle API (requires ~/.kaggle/kaggle.json)
# Uncomment and run if you haven't downloaded yet:
#
# ! kaggle competitions download -c home-credit-default-risk -p ./data
# with zipfile.ZipFile('./data/home-credit-default-risk.zip', 'r') as z:
#     z.extractall('./data')

# Option B: If already downloaded / uploaded, read directly
DATA_DIR = './data'  # Adjust path as needed
df = pd.read_csv(os.path.join(DATA_DIR, 'application_train.csv'))

print(f"Dataset shape: {df.shape}")
display(df.head())

In [ ]:
# --- Step 1.2: Structure & metadata ---
# Separate target from features
TARGET_COL = 'TARGET'
ID_COL = 'SK_ID_CURR'

y = df[TARGET_COL].copy()
X = df.drop(columns=[TARGET_COL, ID_COL])

X.info(verbose=True, show_counts=True)
print(f"\nFeatures shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")
print(f"\nDefault rate: {y.mean():.4f}")

In [ ]:
# --- Step 1.3: Summary statistics & dtypes ---
print("--- Numerical feature statistics ---")
display(X.describe())

print("\n--- Categorical columns ---")
cat_cols = X.select_dtypes(include='object').columns.tolist()
print(f"Count: {len(cat_cols)}")
for col in cat_cols:
    print(f"  {col}: {X[col].nunique()} unique values")

In [ ]:
# --- Step 1.4: Identify and quantify missing values ---
n_rows = len(X)
missing_counts = X.isnull().sum()
missing_pct = (missing_counts / n_rows * 100).round(2)
missing_df = pd.DataFrame({
    'missing_count': missing_counts,
    'missing_pct': missing_pct
})
missing_df = missing_df[missing_df['missing_count'] > 0].sort_values('missing_pct', ascending=False)

print(f"Columns with missing values: {len(missing_df)} / {X.shape[1]}")
display(missing_df.head(20))

# Visualize missingness
fig, ax = plt.subplots(figsize=(12, 4))
top_missing = missing_df.head(20)
ax.barh(top_missing.index, top_missing['missing_pct'])
ax.set_xlabel('Missing %')
ax.set_title('Top 20 Columns by Missing Value %')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# --- Step 1.5: Clean data ---
# Strategy:
#   - Drop columns with > 50% missing (too much imputation bias)
#   - Impute numerical columns with median (robust to outliers)
#   - Impute categorical columns with mode
# TODO: Justify your decisions with evidence from Step 1.4

DROP_THRESHOLD = 0.50

X_clean = X.copy()

# Drop high-missingness columns
cols_to_drop = missing_df[missing_df['missing_pct'] > DROP_THRESHOLD * 100].index.tolist()
if cols_to_drop:
    X_clean = X_clean.drop(columns=cols_to_drop)
    print(f"Dropped {len(cols_to_drop)} columns with >{DROP_THRESHOLD*100:.0f}% missing: {cols_to_drop}")

# Impute remaining missing values
num_cols = X_clean.select_dtypes(include='number').columns.tolist()
cat_cols_clean = X_clean.select_dtypes(include='object').columns.tolist()

for col in num_cols:
    if X_clean[col].isnull().sum() > 0:
        median_val = X_clean[col].median()
        X_clean[col] = X_clean[col].fillna(median_val)

for col in cat_cols_clean:
    if X_clean[col].isnull().sum() > 0:
        mode_val = X_clean[col].mode()[0]
        X_clean[col] = X_clean[col].fillna(mode_val)

assert X_clean.isnull().sum().sum() == 0, "Missing values remain!"
print(f"\n[SUCCESS] Cleaning complete. Shape: {X_clean.shape}. No missing values remaining.")

In [ ]:
# --- Step 1.6: Target variable ---
# TARGET is already binary: 0 = repay, 1 = default (no encoding needed)
y_encoded = y.copy()

print("Target distribution:")
print(y_encoded.value_counts())
print(f"\nDefault rate: {y_encoded.mean():.4f}")

In [ ]:
# --- Step 1.7: Feature engineering ---
# Identify categorical vs numerical columns
CAT_COLS = X_clean.select_dtypes(include='object').columns.tolist()
NUM_COLS = X_clean.select_dtypes(include='number').columns.tolist()

print(f"Categorical columns ({len(CAT_COLS)}): {CAT_COLS}")
print(f"Numerical columns ({len(NUM_COLS)}): {len(NUM_COLS)} total")

# One-hot encode categoricals (drop_first to avoid multicollinearity)
X_encoded = pd.get_dummies(X_clean, columns=CAT_COLS, drop_first=True)

# TODO: Optionally construct new features
# Example: income-to-credit ratio, age in years, etc.
# if 'AMT_INCOME_TOTAL' in X_encoded.columns and 'AMT_CREDIT' in X_encoded.columns:
#     X_encoded['INCOME_CREDIT_RATIO'] = X_encoded['AMT_INCOME_TOTAL'] / (X_encoded['AMT_CREDIT'] + 1e-6)
# if 'DAYS_BIRTH' in X_encoded.columns:
#     X_encoded['AGE_YEARS'] = (-X_encoded['DAYS_BIRTH'] / 365.25).round(0)

print(f"\nEncoded feature shape: {X_encoded.shape}")

In [ ]:
# --- Step 1.8: Train / Validation / Test split (70 / 15 / 15) ---
# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X_encoded, y_encoded, test_size=0.30, random_state=RANDOM_STATE, stratify=y_encoded
)
# Second split: 50% of temp -> 15% val, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")
print(f"Train default rate: {y_train.mean():.4f}")
print(f"Val default rate:   {y_val.mean():.4f}")
print(f"Test default rate:  {y_test.mean():.4f}")

In [ ]:
# --- Step 1.9: Feature scaling (for MLP) ---
# IMPORTANT: Fit scaler on training data ONLY (no data leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrames for interpretability
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=X_val.columns, index=X_val.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print("[SUCCESS] Phase 1 complete.")
print("  - X_train, X_val, X_test (unscaled) -> for GBDT")
print("  - X_train_scaled, X_val_scaled, X_test_scaled -> for MLP")

### Why is feature scaling necessary for MLP but not for GBDT?

**TODO:** Discuss this topic. Key points:
- **MLP (Neural Networks):** Gradient-based optimization is sensitive to feature scales. Large-magnitude features dominate weight updates, causing slow convergence or divergence. Standardization ensures all features contribute equally to learning.
- **GBDT (Tree-based):** Decision trees split on thresholds — they only care about rank order within a feature, not absolute magnitudes. Scaling does not change the split decisions, so trees are invariant to monotonic transformations.

---

## Phase 2: Gradient Boosted Decision Trees (GBDT)

Train an `XGBClassifier` and explore hyperparameters:
- `learning_rate`, `n_estimators`, `max_depth`
- `subsample`, `reg_alpha`, `reg_lambda`

Visualize:
- Training vs validation loss (with `eval_set` and early stopping)
- Feature importance
- Effect of learning rate (at least 3 values)

In [ ]:
# --- Step 2.1: Baseline XGBClassifier ---
xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    use_label_encoder=False,
    eval_metric='logloss'
)

# Train with eval_set for monitoring
t_start = time.time()
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=False
)
xgb_train_time = time.time() - t_start

y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

print(f"XGBoost baseline — Test Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}")
print(f"Training time: {xgb_train_time:.2f}s")

In [ ]:
# --- Step 2.2: Training vs Validation loss curve ---
results = xgb_model.evals_result()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(results['validation_0']['logloss'], label='Train Loss')
ax.plot(results['validation_1']['logloss'], label='Validation Loss')
ax.set_xlabel('Boosting Round')
ax.set_ylabel('Log Loss')
ax.set_title('XGBoost: Training vs Validation Loss')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Step 2.3: Feature importance ---
fig, ax = plt.subplots(figsize=(10, 8))
xgb.plot_importance(xgb_model, ax=ax, max_num_features=20, importance_type='gain')
ax.set_title('XGBoost: Top 20 Feature Importances (Gain)')
plt.tight_layout()
plt.show()

In [ ]:
# --- Step 2.4: Effect of learning rate ---
learning_rates = [0.01, 0.1, 0.3]
lr_results = {}

for lr in learning_rates:
    model = XGBClassifier(
        n_estimators=500,
        learning_rate=lr,
        max_depth=6,
        subsample=0.8,
        random_state=RANDOM_STATE,
        use_label_encoder=False,
        eval_metric='logloss',
        early_stopping_rounds=20
    )
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    lr_results[lr] = {
        'model': model,
        'accuracy': accuracy_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'auc_pr': average_precision_score(y_test, y_proba),
        'best_iteration': model.best_iteration
    }
    print(f"LR={lr}: Accuracy={lr_results[lr]['accuracy']:.4f}, F1={lr_results[lr]['f1']:.4f}, "
          f"AUC-PR={lr_results[lr]['auc_pr']:.4f}, Best iter={lr_results[lr]['best_iteration']}")

# TODO: Add visualization comparing learning rate effects

In [ ]:
# --- Step 2.5: Hyperparameter tuning (optional: GridSearchCV / RandomizedSearchCV) ---
# TODO: Tune max_depth, subsample, reg_alpha, reg_lambda
# Example:
# param_grid = {
#     'max_depth': [3, 6, 9],
#     'subsample': [0.6, 0.8, 1.0],
#     'reg_alpha': [0, 0.1, 1.0],
#     'reg_lambda': [0.5, 1.0, 2.0],
# }
# grid_search = GridSearchCV(
#     XGBClassifier(learning_rate=0.1, n_estimators=200,
#                   random_state=RANDOM_STATE, use_label_encoder=False, eval_metric='logloss'),
#     param_grid, cv=3, scoring='f1', n_jobs=-1, verbose=1
# )
# grid_search.fit(X_train, y_train)
# print(f"Best params: {grid_search.best_params_}")
# print(f"Best F1: {grid_search.best_score_:.4f}")

print("[TODO] Implement hyperparameter tuning.")

In [ ]:
# --- Step 2.6: Final GBDT model (best configuration) ---
# TODO: Retrain best GBDT model with tuned hyperparameters
# For now, use the baseline model
best_xgb = xgb_model  # Replace with tuned model

y_pred_xgb_final = best_xgb.predict(X_test)
y_proba_xgb_final = best_xgb.predict_proba(X_test)[:, 1]

print("\n=== Best GBDT — Test Set Results ===")
print(classification_report(y_test, y_pred_xgb_final, target_names=['Repay (0)', 'Default (1)']))

---

## Phase 3: Multi-Layer Perceptron (MLP)

Train an `MLPClassifier` (scikit-learn) and explore hyperparameters:
- `hidden_layer_sizes` (e.g., (64,), (128, 64), (256, 128, 64))
- `activation` (e.g., relu vs tanh)
- `learning_rate_init` (e.g., 0.001, 0.01, 0.1)
- `max_iter`

Visualize:
- Training loss curve (`loss_curve_`)
- Effect of network depth/width on validation performance

**Note:** MLP uses the **scaled** data (`X_train_scaled`, `X_val_scaled`, `X_test_scaled`).

In [ ]:
# --- Step 3.1: Baseline MLPClassifier ---
mlp_model = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    learning_rate_init=0.001,
    max_iter=300,
    random_state=RANDOM_STATE,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=15,
    verbose=False
)

t_start = time.time()
mlp_model.fit(X_train_scaled, y_train)
mlp_train_time = time.time() - t_start

y_pred_mlp = mlp_model.predict(X_test_scaled)
y_proba_mlp = mlp_model.predict_proba(X_test_scaled)[:, 1]

print(f"MLP baseline — Test Accuracy: {accuracy_score(y_test, y_pred_mlp):.4f}")
print(f"Training time: {mlp_train_time:.2f}s")
print(f"Epochs trained: {mlp_model.n_iter_}")

In [ ]:
# --- Step 3.2: Training loss curve ---
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(mlp_model.loss_curve_, label='Training Loss')
if hasattr(mlp_model, 'validation_scores_') and mlp_model.validation_scores_ is not None:
    ax.plot(mlp_model.validation_scores_, label='Validation Score')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss / Score')
ax.set_title('MLP: Training Loss Curve')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Step 3.3: Effect of network depth/width ---
architectures = [
    (64,),
    (128, 64),
    (256, 128, 64),
]
arch_results = {}

for arch in architectures:
    model = MLPClassifier(
        hidden_layer_sizes=arch,
        activation='relu',
        learning_rate_init=0.001,
        max_iter=300,
        random_state=RANDOM_STATE,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=15,
        verbose=False
    )
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    arch_results[str(arch)] = {
        'model': model,
        'accuracy': accuracy_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'auc_pr': average_precision_score(y_test, y_proba),
        'epochs': model.n_iter_
    }
    print(f"Arch={arch}: Accuracy={arch_results[str(arch)]['accuracy']:.4f}, "
          f"F1={arch_results[str(arch)]['f1']:.4f}, "
          f"AUC-PR={arch_results[str(arch)]['auc_pr']:.4f}, "
          f"Epochs={arch_results[str(arch)]['epochs']}")

# TODO: Add visualization comparing architectures

In [ ]:
# --- Step 3.4: Effect of activation function ---
activations = ['relu', 'tanh']
activation_results = {}

for act in activations:
    model = MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation=act,
        learning_rate_init=0.001,
        max_iter=300,
        random_state=RANDOM_STATE,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=15,
        verbose=False
    )
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    activation_results[act] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'auc_pr': average_precision_score(y_test, y_proba),
    }
    print(f"Activation={act}: Accuracy={activation_results[act]['accuracy']:.4f}, "
          f"F1={activation_results[act]['f1']:.4f}, "
          f"AUC-PR={activation_results[act]['auc_pr']:.4f}")

In [ ]:
# --- Step 3.5: Effect of learning rate ---
mlp_learning_rates = [0.001, 0.01, 0.1]
mlp_lr_results = {}

for lr in mlp_learning_rates:
    model = MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation='relu',
        learning_rate_init=lr,
        max_iter=300,
        random_state=RANDOM_STATE,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=15,
        verbose=False
    )
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    mlp_lr_results[lr] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'auc_pr': average_precision_score(y_test, y_proba),
        'epochs': model.n_iter_
    }
    print(f"LR={lr}: Accuracy={mlp_lr_results[lr]['accuracy']:.4f}, "
          f"F1={mlp_lr_results[lr]['f1']:.4f}, "
          f"AUC-PR={mlp_lr_results[lr]['auc_pr']:.4f}, "
          f"Epochs={mlp_lr_results[lr]['epochs']}")

In [ ]:
# --- Step 3.6: Final MLP model (best configuration) ---
# TODO: Select and retrain best MLP with tuned hyperparameters
# For now, use the baseline model
best_mlp = mlp_model  # Replace with tuned model

y_pred_mlp_final = best_mlp.predict(X_test_scaled)
y_proba_mlp_final = best_mlp.predict_proba(X_test_scaled)[:, 1]

print("\n=== Best MLP — Test Set Results ===")
print(classification_report(y_test, y_pred_mlp_final, target_names=['Repay (0)', 'Default (1)']))

---

## Phase 4: GBDT vs MLP Comparison

Using the **same dataset** and **same train/test split**:

- **(a)** Compare both models on: Accuracy, Precision, Recall, F1-score, AUC-PR
- **(b)** Summary comparison table
- **(c)** Compare training time
- **(d)** Discussion: When to prefer GBDT vs MLP, interpretability, handling of categoricals/missing values, sensitivity to hyperparameters

In [ ]:
# --- Step 4.1: Side-by-side comparison ---
def compute_metrics(name, y_true, y_pred, y_proba, train_time):
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0),
        'AUC-PR': average_precision_score(y_true, y_proba),
        'Train Time (s)': round(train_time, 2)
    }

comparison_results = [
    compute_metrics('XGBoost (GBDT)', y_test, y_pred_xgb_final, y_proba_xgb_final, xgb_train_time),
    compute_metrics('MLP', y_test, y_pred_mlp_final, y_proba_mlp_final, mlp_train_time),
]

comparison_df = pd.DataFrame(comparison_results)
print("\n=== GBDT vs MLP — Comparison Table ===")
display(comparison_df)

In [ ]:
# --- Step 4.2: Confusion matrices ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
labels = ['Repay (0)', 'Default (1)']

for ax, (name, y_pred) in zip(axes, [
    ('XGBoost (GBDT)', y_pred_xgb_final),
    ('MLP', y_pred_mlp_final)
]):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                xticklabels=labels, yticklabels=labels)
    ax.set_title(name)
    ax.set_ylabel('True')
    ax.set_xlabel('Predicted')

plt.suptitle('Confusion Matrices: GBDT vs MLP', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# --- Step 4.3: Precision-Recall curves ---
fig, ax = plt.subplots(figsize=(8, 6))

for name, y_proba in [
    ('XGBoost (GBDT)', y_proba_xgb_final),
    ('MLP', y_proba_mlp_final)
]:
    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)
    ax.plot(rec, prec, label=f'{name} (AUC-PR={ap:.3f})', lw=2)

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves: GBDT vs MLP')
ax.legend(loc='lower left')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Step 4.4: ROC curves ---
fig, ax = plt.subplots(figsize=(8, 6))

for name, y_proba in [
    ('XGBoost (GBDT)', y_proba_xgb_final),
    ('MLP', y_proba_mlp_final)
]:
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', lw=2)

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves: GBDT vs MLP')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Discussion: GBDT vs MLP

**TODO:** Address the following questions:

1. **When would you prefer GBDT over MLP, and vice versa?**
   - *GBDT:* ...
   - *MLP:* ...

2. **How does interpretability differ?**
   - GBDT provides feature importance (gain, weight, cover) — directly interpretable.
   - MLP is a "black box" — weights are distributed across layers, making it hard to trace individual feature contributions.

3. **How does each model handle categorical features and missing values?**
   - GBDT (XGBoost) natively handles missing values by learning optimal split directions. Can handle label-encoded categoricals without one-hot.
   - MLP requires explicit preprocessing: missing value imputation, one-hot encoding, and feature scaling.

4. **Which model is more sensitive to hyperparameter choices?**
   - ...

---

## AI Tool Disclosure

**TODO:** List any AI tools used and their roles. Clarify which parts were your own contribution.

- ...